# CardioFusion-XAI ? Biomarker SHAP explainability

This notebook explains one held-out test sample with the selected, source-specific biomarker model. It uses the matching training-only imputer, same-source background rows, and the saved split manifest. The example is for research and is not a clinical report or causal explanation.

In [9]:
from pathlib import Path
import os
import sys


def find_ml_service_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir() and (candidate / "config").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate ml-service/. Start Jupyter from the repository or from ml-service/."
    )


ML_ROOT = find_ml_service_root()
os.chdir(ML_ROOT)
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))
print(f"ML project root: {ML_ROOT}")

ML project root: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service


## 1. Select a trained source expert and a test record

In [10]:
import hashlib
import json
from types import SimpleNamespace

import joblib
import numpy as np
import pandas as pd

from src.calibration.calibrator import PlattCalibrator
from src.normalization.biomarker_adapter import BiomarkerInput, SOURCE_SCHEMAS
from src.training.biomarker_training_data import load_biomarker_source
from src.xai.explanation_service import ExplanationService

SOURCE = "zheen"  # zheen, framingham, uci_heart_failure
DATA_ROOT = ML_ROOT / "data/raw/biomarkers"
TRAINING_ROOT = ML_ROOT / "artifacts/03_training/biomarkers/selected_gbdt_stacking/v1"
SOURCE_MODELS = ML_ROOT / "checkpoints/biomarkers" / SOURCE
if SOURCE not in SOURCE_SCHEMAS:
    raise ValueError(f"No selected inference schema is configured for {SOURCE!r}.")
TARGET = SOURCE_SCHEMAS[SOURCE][1]
MODEL_PATH = SOURCE_MODELS / TARGET / "selected_gbdt_stacking/v1/model.joblib"
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Selected model is missing: {MODEL_PATH}. Run the selected biomarker training notebook first.")

payload = joblib.load(MODEL_PATH)
if payload.get("target") != TARGET:
    raise ValueError(f"Model target mismatch: expected {TARGET!r}, found {payload.get('target')!r}.")
feature_names = tuple(payload.get("feature_names", ()))
model = payload.get("model")
if not feature_names or model is None or not callable(getattr(model, "predict_proba", None)):
    raise ValueError("Selected model artifact is missing its feature schema or predict_proba model.")
background = np.asarray(payload.get("background_features"), dtype=np.float32)
if background.ndim != 2 or background.shape[1] != len(feature_names) or background.shape[0] < 2:
    raise ValueError("Selected model artifact has no compatible same-source training background.")

# Load the calibrator only if the manifest ties it to this exact checkpoint.
model_hash = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
manifest_path = ML_ROOT / "checkpoints/calibration/selected_models/v1/manifest.json"
calibration_manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.is_file() else {}
calibration_entry = calibration_manifest.get("biomarkers", {}).get(SOURCE, {}).get(TARGET, {})
calibrator = None
if calibration_entry.get("status") == "platt":
    if calibration_entry.get("base_model_sha256") != model_hash:
        raise ValueError("Calibration artifact was fitted for a different model checkpoint. Rerun selected-model calibration.")
    calibration_path = ML_ROOT / "checkpoints/calibration" / calibration_entry["artifact"]
    if not calibration_path.is_file():
        raise FileNotFoundError(f"Matching calibration artifact is missing: {calibration_path}")
    calibrator = PlattCalibrator.load(calibration_path)

# ExplanationService needs the selected expert and its optional verified calibrator.
models = SimpleNamespace(
    biomarker_experts={SOURCE: {
        "model": model,
        "feature_names": feature_names,
        "background_features": background,
        "selected_model": str(payload.get("selected_model", type(model).__name__)),
    }},
    biomarker_expert_calibrators={SOURCE: calibrator} if calibrator is not None else {},
)
explanation_service = ExplanationService(models)
print("Source:", SOURCE)
print("Target:", TARGET)
print("Selected model:", models.biomarker_experts[SOURCE]["selected_model"])
print("Calibration:", "Platt" if calibrator is not None else "uncalibrated")
print("Checkpoint SHA-256:", model_hash)

Source: zheen
Target: acute_mi
Selected model: HistGradientBoosting
Calibration: Platt
Checkpoint SHA-256: 9f392edd23d48c779bcedb79dbcc696d4adf1c9ea4925444b99aa70e91dcc291


## 2. Load a held-out source-specific sample and make an exact-order input

In [11]:
data = load_biomarker_source(DATA_ROOT, SOURCE)
if TARGET not in data.targets:
    raise ValueError(f"Dataset {SOURCE!r} does not contain the selected endpoint {TARGET!r}.")
if tuple(data.features.columns) != feature_names:
    raise ValueError("Dataset feature order does not match the selected model artifact.")

# Select a held-out record from the exact split used to create the selected model predictions.
split_path = TRAINING_ROOT / "split_manifest.csv"
if not split_path.is_file():
    raise FileNotFoundError(f"Selected-model split manifest is missing: {split_path}. Run the selected biomarker training notebook first.")
splits = pd.read_csv(split_path, dtype={"record_id": str})
held_out = splits.loc[
    (splits["dataset"] == SOURCE)
    & (splits["target"] == TARGET)
    & (splits["split"] == "test"),
    "record_id",
]
if held_out.empty:
    raise RuntimeError(f"No held-out test records found for {SOURCE}/{TARGET} in {split_path}.")
record_id = str(held_out.iloc[0])
prefix, separator, row_index_text = record_id.partition(":")
if not separator or prefix != SOURCE or not row_index_text.isdigit():
    raise ValueError(f"Unexpected selected-model record ID format: {record_id!r}.")
row_index = int(row_index_text)
if row_index >= len(data.features):
    raise IndexError(f"Record {record_id!r} is outside the loaded dataset.")

row = data.features.iloc[row_index].reindex(feature_names)
row_frame = pd.DataFrame([row.to_numpy()], columns=feature_names)
# Apply the exact fitted training imputer so both inference and SHAP receive finite values.
imputer = getattr(model, "named_steps", {}).get("imputer")
if imputer is None:
    raise ValueError("Selected model pipeline does not contain its fitted training imputer.")
values = np.asarray(imputer.transform(row_frame), dtype=np.float32).reshape(-1)
if not np.isfinite(values).all():
    raise ValueError("Selected held-out sample remains non-finite after the fitted imputer.")
input_item = BiomarkerInput(
    source=SOURCE,
    values=values,
    feature_names=feature_names,
    task=TARGET,
    missing_features=(),
)
print("Held-out sample:", record_id)
print("Task candidates:", list(data.targets))
print("Feature count:", len(input_item.feature_names))

Held-out sample: zheen:393
Task candidates: ['acute_mi']
Feature count: 8


## 3. Generate prediction and SHAP explanation

The explainer uses `background_features.npy`, saved from the source's training partition after the training-only imputer and scaler. If SHAP is unavailable or an operator is unsupported, the cell reports the actual exception rather than fabricating an explanation.

In [12]:
# Match the inference path: positive-class probability, followed by the verified calibrator if available.
classes = list(model.classes_)
if 1 not in classes:
    raise ValueError("Selected model has no positive class 1.")
input_frame = pd.DataFrame([input_item.values], columns=feature_names)
raw_probability = float(model.predict_proba(input_frame)[0, classes.index(1)])
probability = float(calibrator.predict_proba([raw_probability])[0]) if calibrator is not None else raw_probability
prediction = {
    TARGET: probability,
    f"{TARGET}_raw": raw_probability,
    "_metadata": {
        "model": models.biomarker_experts[SOURCE]["selected_model"],
        "targets": {TARGET: {"calibrated": calibrator is not None,
                             "calibration_method": "platt" if calibrator is not None else None}},
    },
}
print("Model outputs:")
print(prediction)

explanation = explanation_service.explain_biomarker(input_item)
print("\nSHAP explanation:")
print(explanation)

Model outputs:
{'acute_mi': 0.9995343123762176, 'acute_mi_raw': 0.999367129187998, '_metadata': {'model': 'HistGradientBoosting', 'targets': {'acute_mi': {'calibrated': True, 'calibration_method': 'platt'}}}}

SHAP explanation:
{'dataset': 'zheen', 'target': 'acute_mi', 'calibration_status': 'platt', 'method': 'SHAP_PermutationExplainer', 'output': 0.9995343123762176, 'base_value': 0.7342076958667809, 'additivity_residual': 5.551115123125783e-16, 'feature_contributions': [{'feature': 'ck_mb', 'shap_value': 0.13286102352828522}, {'feature': 'troponin', 'shap_value': 0.13093203701331246}, {'feature': 'gender', 'shap_value': -0.006022816960857047}, {'feature': 'age', 'shap_value': 0.004335606253369817}, {'feature': 'heart_rate', 'shap_value': 0.003353964818359423}, {'feature': 'blood_glucose', 'shap_value': -0.0005104980500835188}, {'feature': 'diastolic_bp', 'shap_value': 0.0002197233463468451}, {'feature': 'systolic_bp', 'shap_value': 0.00015757656070292114}], 'background_size': 32, 'wa

## 4. Review the feature attribution

The table ranks features by mean absolute SHAP value for the selected sample and scalar binary endpoint. Sign and magnitude are relative to the fitted model, selected background distribution and output function. They do not establish that a feature causes a cardiovascular event.

In [13]:
contributions = explanation.get("feature_contributions", [])
shap_table = pd.DataFrame(contributions)
if shap_table.empty:
    raise RuntimeError("The SHAP explainer returned no feature contributions.")
display(shap_table.assign(absolute_shap=shap_table["shap_value"].abs()).sort_values("absolute_shap", ascending=False).drop(columns="absolute_shap"))
print("Model output:", explanation["output"])
print("Baseline output:", explanation["base_value"])
print("Additivity residual:", explanation["additivity_residual"])


,feature,shap_value
0,ck_mb,0.132861
1,troponin,0.130932
2,gender,-0.006023
3,age,0.004336
4,heart_rate,0.003354
5,blood_glucose,-0.000510
6,diastolic_bp,0.000220
7,systolic_bp,0.000158


Model output: 0.9995343123762176
Baseline output: 0.7342076958667809
Additivity residual: 5.551115123125783e-16


## 5. Reproducibility and limitations

Record the source dataset version, target, sample row identifier (without leaking personal identifiers), checkpoint file/hash, feature manifest, background dataset and SHAP package version in the research log. SHAP is a model attribution method, not a causal method and not proof of clinical plausibility.